# YouTube AI-Slop: transcript-routed synthetic-voice experiment

This Colab notebook tests whether an AI-writing detector can **route** expensive synthetic-voice inference toward useful parts of a video.

It does not treat AI-written text as proof of AI voice. It also samples uniform control clips, because synthetic speech can read human-written text and people can read AI-written scripts.

> **Runtime:** select **Runtime → Change runtime type → T4 GPU** before running.  
> **License:** the Arena-hosted audio detector is CC BY-NC-SA 4.0. Use this notebook for evaluation/prototyping only unless you separately resolve model licensing.

## Goal

Produce inspectable, clip-level evidence:

- timestamped transcript;
- AI-writing scores over overlapping 80–150-word regions;
- 4-second synthetic-voice scores from text-ranked and uniform-control clips; and
- conservative video-level aggregates for comparing experiments.

The output is not a production block decision.

## Setup

Install the Colab dependencies. PyTorch and FFmpeg are provided by Colab.

In [ ]:
from pathlib import Path
import subprocess
import sys

REQUIREMENTS_URL = "https://raw.githubusercontent.com/emilzmmn04/youtube-ai-slop-audio-detector/main/requirements-colab.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", REQUIREMENTS_URL])
print("Dependencies installed. If Colab asks for a runtime restart, restart and continue here.")

## Configuration

Start with the defaults. Raising sample counts improves coverage but increases GPU time.

In [ ]:
# A YouTube URL is required for captions. If audio download is blocked, upload the same media when prompted.
YOUTUBE_URL = ""  # @param {type:"string"}

# Models
TEXT_MODEL_ID = "GeorgeDrayson/modernbert-ai-detection-raid-mage"
AUDIO_MODEL_ID = "SpeechAntiSpoofingBenchmarks/Wav2Vec2-Small-AntiDeepfake-NDA"

# Transcript routing
TEXT_WINDOW_WORDS = 120  # @param {type:"integer"}
TEXT_STRIDE_WORDS = 60  # @param {type:"integer"}
TOP_TEXT_WINDOWS = 3  # @param {type:"integer"}
CLIPS_PER_TEXT_WINDOW = 3  # @param {type:"integer"}
UNIFORM_CONTROL_CLIPS = 6  # @param {type:"integer"}

# The Arena ONNX export uses exactly 64,000 samples at 16 kHz (4 seconds).
SAMPLE_RATE = 16_000
AUDIO_CLIP_SAMPLES = 64_000
AUDIO_CLIP_SECONDS = AUDIO_CLIP_SAMPLES / SAMPLE_RATE
PROVISIONAL_AUDIO_THRESHOLD = 0.80  # @param {type:"number"}

assert 40 <= TEXT_WINDOW_WORDS <= 500
assert 1 <= TEXT_STRIDE_WORDS <= TEXT_WINDOW_WORDS
assert TOP_TEXT_WINDOWS >= 1
print(f"Audio clips: {AUDIO_CLIP_SECONDS:.4f} seconds each")

## Step 1 — Acquire and normalize media

The notebook first tries `yt-dlp`. If YouTube blocks Colab's datacenter IP, it opens a file uploader while preserving the URL for caption retrieval. All input is converted to 16 kHz mono WAV.

In [ ]:
from pathlib import Path

WORKDIR = Path("/content/ai_slop_detector")
WORKDIR.mkdir(parents=True, exist_ok=True)
SOURCE_PATH = WORKDIR / "input_media"
AUDIO_PATH = WORKDIR / "input_audio.wav"

source_media = None
if YOUTUBE_URL.strip():
    output_template = str(WORKDIR / "download.%(ext)s")
    command = [
        "yt-dlp", "--no-playlist", "-f", "bestaudio/best",
        "-o", output_template, YOUTUBE_URL.strip(),
    ]
    try:
        subprocess.run(command, check=True)
        downloaded = [p for p in WORKDIR.glob("download.*") if p.is_file()]
        if downloaded:
            source_media = downloaded[0]
    except subprocess.CalledProcessError:
        print("YouTube blocked the Colab downloader; upload the same video's audio/video below.")

if source_media is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("File upload requires Colab; alternatively set YOUTUBE_URL") from exc
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded")
    uploaded_name = next(iter(uploaded))
    source_media = WORKDIR / Path(uploaded_name).name
    source_media.write_bytes(uploaded[uploaded_name])

subprocess.run([
    "ffmpeg", "-y", "-i", str(source_media), "-vn",
    "-ac", "1", "-ar", str(SAMPLE_RATE), "-c:a", "pcm_s16le", str(AUDIO_PATH),
], check=True, capture_output=True)

print(f"Normalized audio: {AUDIO_PATH}")
print(f"Size: {AUDIO_PATH.stat().st_size / 1_000_000:.1f} MB")

## Step 2 — Fetch YouTube captions with timestamps

The notebook uses the video's existing YouTube caption track instead of running ASR. Caption-segment times are interpolated across words so text windows can be mapped back to audio.

In [ ]:
import html
import json
import re
from urllib.parse import parse_qs, urlparse
from youtube_transcript_api import YouTubeTranscriptApi


def youtube_video_id(url):
    parsed = urlparse(url.strip())
    if parsed.netloc in {"youtu.be", "www.youtu.be"}:
        return parsed.path.strip("/").split("/")[0]
    if parsed.netloc.endswith("youtube.com"):
        if parsed.path == "/watch":
            return parse_qs(parsed.query).get("v", [None])[0]
        parts = parsed.path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] in {"shorts", "embed", "live"}:
            return parts[1]
    return None


VIDEO_ID = youtube_video_id(YOUTUBE_URL)
if not VIDEO_ID:
    raise ValueError("Set YOUTUBE_URL to a valid YouTube video so captions can be fetched.")

api = YouTubeTranscriptApi()
tracks = list(api.list(VIDEO_ID))
if not tracks:
    raise RuntimeError("This video exposes no YouTube caption tracks.")
tracks.sort(key=lambda track: (
    not str(track.language_code).lower().startswith("en"),
    bool(track.is_generated),
))
track = tracks[0]
captions = track.fetch()

words = []
full_text = []
for item in captions:
    clean_text = html.unescape(re.sub(r"<[^>]+>", " ", item.text)).replace("\n", " ")
    tokens = clean_text.split()
    if not tokens:
        continue
    start_s = float(item.start)
    duration_s = max(float(item.duration), 0.01)
    full_text.extend(tokens)
    for index, token in enumerate(tokens):
        words.append({
            "text": token,
            "start_s": start_s + duration_s * index / len(tokens),
            "end_s": start_s + duration_s * (index + 1) / len(tokens),
        })

if not words:
    raise RuntimeError("The selected YouTube caption track was empty.")

transcript_record = {
    "video_id": VIDEO_ID,
    "source": "youtube_captions",
    "language": track.language_code,
    "is_generated": bool(track.is_generated),
    "text": " ".join(full_text),
    "words": words,
}
(WORKDIR / "transcript.json").write_text(json.dumps(transcript_record, indent=2), encoding="utf-8")
print(
    f"YouTube captions: {len(words):,} timestamped words, "
    f"{words[-1]['end_s'] / 60:.1f} min, "
    f"language={track.language_code}, auto_generated={track.is_generated}"
)
print(transcript_record["text"][:800])

## Step 3 — Build and score transcript windows

AI-writing classifiers are unreliable on tiny snippets, so the routing stage scores overlapping long windows. The text score ranks regions; it is not treated as a factual probability or blocking signal.

In [ ]:
import gc
import time
from pathlib import Path
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import requests
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def build_text_windows(word_records, window_words, stride_words):
    rows = []
    total = len(word_records)
    starts = list(range(0, max(total - window_words + 1, 1), stride_words))
    final_start = max(total - window_words, 0)
    if not starts or starts[-1] != final_start:
        starts.append(final_start)
    for window_id, start_index in enumerate(sorted(set(starts))):
        end_index = min(start_index + window_words, total)
        window = word_records[start_index:end_index]
        rows.append({
            "window_id": window_id,
            "start_word": start_index,
            "end_word": end_index,
            "start_s": window[0]["start_s"],
            "end_s": window[-1]["end_s"],
            "word_count": len(window),
            "text": " ".join(item["text"] for item in window),
        })
    return pd.DataFrame(rows)


def download_hf_large_file(repo_id, filename, destination, attempts=20, chunk_size=8 * 1024 * 1024):
    destination = Path(destination)
    signed_url = None
    for attempt in range(attempts):
        resolve_url = (
            f"https://huggingface.co/{repo_id}/resolve/main/{filename}"
            f"?download=true&fresh={time.time_ns()}"
        )
        redirect = requests.get(resolve_url, allow_redirects=False, headers={"Cache-Control": "no-cache"}, timeout=30)
        redirect.raise_for_status()
        candidate = redirect.headers.get("location")
        if not candidate:
            continue
        probe = requests.get(candidate, headers={"Range": "bytes=0-0"}, timeout=30)
        if probe.status_code == 206:
            signed_url = candidate
            total = int(probe.headers["content-range"].split("/")[-1])
            print(f"CDN {urlparse(candidate).netloc} accepted on attempt {attempt + 1}")
            break
    if signed_url is None:
        raise RuntimeError(f"No working signed URL for {repo_id}/{filename}")
    if destination.exists() and destination.stat().st_size == total:
        return destination
    with destination.open("wb") as handle:
        for number, start in enumerate(range(0, total, chunk_size), start=1):
            end = min(start + chunk_size - 1, total - 1)
            part = requests.get(signed_url, headers={"Range": f"bytes={start}-{end}"}, timeout=(30, 300))
            if part.status_code != 206:
                raise RuntimeError(f"Range {start}-{end} returned {part.status_code}")
            handle.write(part.content)
            if number % 10 == 0 or end == total - 1:
                print(f"downloaded {end + 1:,}/{total:,} bytes", flush=True)
    return destination


text_model_dir = WORKDIR / "text_model"
text_model_dir.mkdir(parents=True, exist_ok=True)
for filename in ["config.json", "tokenizer_config.json", "tokenizer.json", "special_tokens_map.json"]:
    response = requests.get(f"https://huggingface.co/{TEXT_MODEL_ID}/resolve/main/{filename}?download=true", timeout=30)
    response.raise_for_status()
    (text_model_dir / filename).write_bytes(response.content)
download_hf_large_file(TEXT_MODEL_ID, "model.safetensors", text_model_dir / "model.safetensors")

text_windows = build_text_windows(words, TEXT_WINDOW_WORDS, TEXT_STRIDE_WORDS)
tokenizer = AutoTokenizer.from_pretrained(text_model_dir)
text_model = AutoModelForSequenceClassification.from_pretrained(text_model_dir).to("cuda").eval()

scores = []
batch_size = 8
for offset in range(0, len(text_windows), batch_size):
    batch_text = text_windows.iloc[offset:offset + batch_size]["text"].tolist()
    inputs = tokenizer(batch_text, padding=True, truncation=True, max_length=1024, return_tensors="pt")
    inputs = {name: value.to("cuda") for name, value in inputs.items()}
    with torch.inference_mode():
        logits = text_model(**inputs).logits.float()
    if logits.shape[-1] == 1:
        batch_scores = torch.sigmoid(logits[:, 0])
    else:
        # The model's research usage defines class index 1 as machine-generated.
        batch_scores = torch.softmax(logits, dim=-1)[:, 1]
    scores.extend(batch_scores.cpu().tolist())

text_windows["ai_text_score"] = scores
text_windows["text_rank"] = text_windows["ai_text_score"].rank(method="first", ascending=False).astype(int)
text_windows.sort_values("text_rank").to_csv(WORKDIR / "transcript_windows.csv", index=False)
display(text_windows.sort_values("text_rank").head(10)[
    ["text_rank", "start_s", "end_s", "word_count", "ai_text_score", "text"]
])

del text_model, tokenizer, inputs, logits
gc.collect()
torch.cuda.empty_cache()

## Step 4 — Choose text-ranked and control audio clips

Each candidate is exactly the duration consumed by the Arena detector. Uniform controls preserve coverage outside AI-looking transcript regions. Near-duplicate timestamps are removed.

In [ ]:
def evenly_spaced(values, count):
    if not values or count <= 0:
        return []
    indexes = np.linspace(0, len(values) - 1, min(count, len(values))).round().astype(int)
    return [values[index] for index in sorted(set(indexes.tolist()))]


def nearest_text_score(center_s):
    overlapping = text_windows[(text_windows.start_s <= center_s) & (text_windows.end_s >= center_s)]
    if len(overlapping):
        return float(overlapping.ai_text_score.max())
    nearest_index = ((text_windows.start_s + text_windows.end_s) / 2 - center_s).abs().idxmin()
    return float(text_windows.loc[nearest_index, "ai_text_score"])


candidates = []
top_windows = text_windows.nsmallest(TOP_TEXT_WINDOWS, "text_rank")
for row in top_windows.itertuples():
    region_words = words[row.start_word:row.end_word]
    centers = [(item["start_s"] + item["end_s"]) / 2 for item in region_words]
    for center_s in evenly_spaced(centers, CLIPS_PER_TEXT_WINDOW):
        candidates.append({
            "source": "text_ranked",
            "center_s": center_s,
            "ai_text_score": float(row.ai_text_score),
            "text_rank": int(row.text_rank),
        })

all_speech_centers = [(item["start_s"] + item["end_s"]) / 2 for item in words]
for center_s in evenly_spaced(all_speech_centers, UNIFORM_CONTROL_CLIPS):
    candidates.append({
        "source": "uniform_control",
        "center_s": center_s,
        "ai_text_score": nearest_text_score(center_s),
        "text_rank": None,
    })

deduplicated = []
for candidate in sorted(candidates, key=lambda item: (item["center_s"], item["source"])):
    if any(abs(candidate["center_s"] - prior["center_s"]) < AUDIO_CLIP_SECONDS / 2 for prior in deduplicated):
        continue
    start_s = max(0.0, candidate["center_s"] - AUDIO_CLIP_SECONDS / 2)
    candidate["start_s"] = start_s
    candidate["end_s"] = start_s + AUDIO_CLIP_SECONDS
    deduplicated.append(candidate)

candidate_clips = pd.DataFrame(deduplicated)
display(candidate_clips)
print(candidate_clips["source"].value_counts())

## Step 5 — Score clips with an Arena anti-deepfake model

The Arena-hosted Wav2Vec2 Small AntiDeepfake NDA model consumes 4 seconds of 16 kHz waveform. Class 0 is fake and class 1 is real. `spoof_score` is its softmax fake score, not a calibrated probability.

In [ ]:
import librosa
import onnxruntime as ort

audio_model_dir = WORKDIR / "audio_model"
audio_model_dir.mkdir(parents=True, exist_ok=True)
onnx_name = "wav2vec2-small-antideepfake-nda.onnx"
onnx_path = download_hf_large_file(AUDIO_MODEL_ID, onnx_name, audio_model_dir / onnx_name)

waveform, loaded_rate = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
assert loaded_rate == SAMPLE_RATE
audio_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
input_meta = audio_session.get_inputs()[0]
input_name = input_meta.name
required_samples = int(input_meta.shape[1])
print("ONNX input:", input_name, input_meta.shape)


def extract_fixed_clip(start_s):
    start_sample = max(0, int(round(start_s * SAMPLE_RATE)))
    clip = waveform[start_sample:start_sample + required_samples]
    if len(clip) == 0:
        raise ValueError(f"No audio available at {start_s:.2f}s")
    if len(clip) < required_samples:
        repeats = int(np.ceil(required_samples / len(clip)))
        clip = np.tile(clip, repeats)[:required_samples]
    return clip.astype(np.float32, copy=False)


spoof_scores = []
predicted_labels = []
for row in candidate_clips.itertuples():
    clip = extract_fixed_clip(row.start_s)[None, :]
    logits = np.asarray(audio_session.run(None, {input_name: clip})[0], dtype=np.float64)[0]
    probabilities = np.exp(logits - logits.max())
    probabilities /= probabilities.sum()
    spoof_score = float(probabilities[0])
    spoof_scores.append(spoof_score)
    predicted_labels.append("spoof" if spoof_score >= 0.5 else "bonafide")

candidate_clips["end_s"] = candidate_clips["start_s"] + required_samples / SAMPLE_RATE
candidate_clips["spoof_score"] = spoof_scores
candidate_clips["model_label"] = predicted_labels
candidate_clips["above_provisional_threshold"] = (
    candidate_clips["spoof_score"] >= PROVISIONAL_AUDIO_THRESHOLD
)
candidate_clips.sort_values("spoof_score", ascending=False).to_csv(WORKDIR / "clip_scores.csv", index=False)

display(candidate_clips.sort_values("spoof_score", ascending=False))

del audio_session, waveform
gc.collect()

## Checks — Aggregate without overclaiming

The median of the three highest audio scores is less sensitive to one anomalous clip than the maximum. Requiring multiple positive clips is a provisional experiment rule, not a calibrated YouTube blocker.

In [ ]:
sorted_audio_scores = sorted(candidate_clips["spoof_score"].tolist(), reverse=True)
top_count = min(3, len(sorted_audio_scores))
audio_peak_median = float(np.median(sorted_audio_scores[:top_count]))
evidence_count = int(candidate_clips["above_provisional_threshold"].sum())

summary = {
    "transcript_source": transcript_record["source"],
    "text_model": TEXT_MODEL_ID,
    "audio_model": AUDIO_MODEL_ID,
    "duration_minutes": round(words[-1]["end_s"] / 60, 3),
    "timestamped_words": len(words),
    "text_windows": len(text_windows),
    "clips_tested": len(candidate_clips),
    "text_ranked_clips": int((candidate_clips.source == "text_ranked").sum()),
    "uniform_control_clips": int((candidate_clips.source == "uniform_control").sum()),
    "max_ai_text_score": float(text_windows.ai_text_score.max()),
    "max_spoof_score": float(candidate_clips.spoof_score.max()),
    "median_top_3_spoof_score": audio_peak_median,
    "provisional_audio_threshold": PROVISIONAL_AUDIO_THRESHOLD,
    "clips_above_provisional_threshold": evidence_count,
    "provisional_multi_clip_evidence": evidence_count >= 2,
    "warning": "Experimental, uncalibrated evidence. Do not use as an automatic block decision.",
}
(WORKDIR / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(pd.Series(summary, name="value").to_frame())

## Inspect and export

Listen to the highest-scoring clips before interpreting the result. Music, overlapping speakers, silence, compression, and ASR errors are useful failure cases to record.

In [ ]:
from IPython.display import Audio, display

# Reload only for listening; this does not run either model.
listen_waveform, _ = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
for row in candidate_clips.nlargest(min(5, len(candidate_clips)), "spoof_score").itertuples():
    start = int(row.start_s * SAMPLE_RATE)
    end = start + AUDIO_CLIP_SAMPLES
    print(f"{row.start_s:.2f}s | {row.source} | spoof={row.spoof_score:.3f} | text={row.ai_text_score:.3f}")
    display(Audio(listen_waveform[start:end], rate=SAMPLE_RATE))

print("Artifacts:")
for artifact in ["transcript.json", "transcript_windows.csv", "clip_scores.csv", "run_summary.json"]:
    print(WORKDIR / artifact)

# Optional download:
# from google.colab import files
# for artifact in ["transcript.json", "transcript_windows.csv", "clip_scores.csv", "run_summary.json"]:
#     files.download(str(WORKDIR / artifact))

## Next steps

Run the same labeled videos under three policies:

1. uniform audio clips only;
2. text-ranked clips only; and
3. text-ranked plus uniform controls (this notebook).

Evaluate at the **video level** with complete videos/channels held out. Record precision, recall, AUROC, and false blocks per 100 hours. Repeat after YouTube-like AAC/Opus re-encoding. Only after collecting those results should the transcript and audio evidence be calibrated into a combined classifier.